# Revisión y Reproyección Masiva de Capas (SHP y Raster)

**Objetivo:** Recorrer una carpeta raíz (y todas sus subcarpetas), identificar el sistema de referencia (EPSG) de cada **shapefile** y **raster**, y reproyectar los que no estén en el EPSG objetivo, **sobrescribiendo el archivo en la misma ubicación y con el mismo nombre**.

**Flujo:**
1. **Inventario:** lista cada capa con su CRS, código EPSG y estado (`OK`, `Reproyectar`, `Sin CRS`, `Error`).
2. **Respaldo:** antes de sobrescribir, copia el archivo original (con todos sus archivos asociados) a una carpeta de respaldo que replica la estructura de carpetas.
3. **Reproyección:** reescribe la capa en el EPSG objetivo.
4. **Verificación:** vuelve a hacer el inventario para confirmar el resultado.

> ⚠️ **La reproyección sobrescribe los archivos originales.** Por seguridad, el notebook arranca en modo `SOLO_INVENTARIO = True` (no modifica nada). Revisa el inventario y luego cambia a `False` para reproyectar.

**Notas:**
- Las capas **sin CRS definido** no se reproyectan (no se puede saber su sistema de origen); se reportan para revisión manual.
- Rasters: se usa remuestreo `nearest` para datos enteros/categóricos y `bilinear` para datos continuos (flotantes).
- Se eliminan los archivos auxiliares que quedarían desactualizados (índices espaciales `.sbn/.sbx/.qix`, `.aux.xml`, `.ovr`, archivos *world* `.tfw`, etc.). Quedan en el respaldo.

## 1. Importaciones

In [ ]:
import os
import shutil
import warnings
from datetime import datetime
from pathlib import Path

# PostgreSQL/PostGIS definen PROJ_LIB y GDAL_DATA a nivel sistema apuntando a versiones
# incompatibles con rasterio ("proj.db ... comes from another PROJ installation").
# Se quitan solo para este kernel (antes de importar rasterio) para que use sus propios datos.
for var in ('PROJ_LIB', 'PROJ_DATA', 'GDAL_DATA'):
    os.environ.pop(var, None)

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import rasterio
import rasterio.shutil
from rasterio.warp import calculate_default_transform, reproject, Resampling
from pyproj import CRS

warnings.filterwarnings('ignore')

print(f'GeoPandas: {gpd.__version__}')
print(f'Rasterio : {rasterio.__version__} (GDAL {rasterio.__gdal_version__})')

## 2. Parámetros

In [ ]:
# ─── AJUSTA ESTOS VALORES ────────────────────────────────────────────────────

RUTA_RAIZ = r'RUTA/A/LA/CARPETA/RAIZ'     # <-- PLACEHOLDER: carpeta con las subcarpetas de capas

EPSG_OBJETIVO = 6372                      # EPSG al que deben quedar todas las capas

# True: solo hace el inventario (no modifica nada). False: reproyecta y sobrescribe.
SOLO_INVENTARIO = True

# Copia los originales antes de sobrescribirlos (muy recomendado)
HACER_RESPALDO   = True
CARPETA_RESPALDO = '_respaldo_reproyeccion'   # se crea dentro de RUTA_RAIZ

EXT_VECTOR = ['.shp']
EXT_RASTER = ['.tif', '.tiff', '.img', '.asc', '.jp2']

# Remuestreo de rasters según tipo de dato
RESAMPLING_ENTEROS   = Resampling.nearest    # categóricos (usos de suelo, clases, máscaras)
RESAMPLING_FLOTANTES = Resampling.bilinear   # continuos (elevación, temperatura, índices)

# Codificación para leer los .dbf sin archivo .cpg (None = la que detecte GDAL)
ENCODING_SHP = None

# Reporte CSV del inventario
OUTPUT_DIR = 'datos/reporte_crs'

# ─────────────────────────────────────────────────────────────────────────────

CRS_OBJETIVO = CRS.from_epsg(EPSG_OBJETIVO)
RAIZ = Path(RUTA_RAIZ)
if not RAIZ.is_dir():
    raise FileNotFoundError(f'No existe la carpeta: {RAIZ}')

print(f'Carpeta raíz    : {RAIZ}')
print(f'EPSG objetivo   : {EPSG_OBJETIVO} — {CRS_OBJETIVO.name}')
print(f'Solo inventario : {SOLO_INVENTARIO}')
print(f'Respaldo        : {HACER_RESPALDO} ({RAIZ / CARPETA_RESPALDO})')

## 3. Funciones de identificación de CRS

Para cada capa se lee el CRS y se intenta identificar su código EPSG. Los `.prj` en formato ESRI no siempre traen el código explícito; en ese caso se busca el EPSG equivalente en la base de datos de PROJ.
Una capa se considera `OK` si su EPSG coincide con el objetivo o si su CRS es equivalente al objetivo aunque no se haya podido identificar el código.

In [ ]:
def leer_crs(ruta, tipo):
    '''Devuelve el CRS (pyproj) de la capa, o None si no tiene.'''
    if tipo == 'vector':
        crs = pyogrio.read_info(ruta)['crs']
        return CRS.from_user_input(crs) if crs else None
    with rasterio.open(ruta) as src:
        return CRS.from_wkt(src.crs.to_wkt()) if src.crs else None


def identificar_epsg(crs):
    '''Código EPSG del CRS; prueba con confianza decreciente para .prj de ESRI.'''
    for confianza in (70, 50, 25):
        epsg = crs.to_epsg(min_confidence=confianza)
        if epsg is not None:
            return epsg
    return None


def es_objetivo(crs, epsg):
    return epsg == EPSG_OBJETIVO or crs.equals(CRS_OBJETIVO, ignore_axis_order=True)


def inventariar(raiz):
    registros = []
    for carpeta, subcarpetas, archivos in os.walk(raiz):
        # no recorrer la carpeta de respaldo ni temporales
        subcarpetas[:] = [d for d in subcarpetas
                          if d != CARPETA_RESPALDO and not d.startswith('.tmp_reproy')]
        for nombre in sorted(archivos):
            ext = Path(nombre).suffix.lower()
            if ext in EXT_VECTOR:
                tipo = 'vector'
            elif ext in EXT_RASTER:
                tipo = 'raster'
            else:
                continue
            ruta = Path(carpeta) / nombre
            reg = {'carpeta': str(Path(carpeta).relative_to(raiz)), 'archivo': nombre,
                   'tipo': tipo, 'ruta': str(ruta),
                   'crs_nombre': None, 'epsg': None, 'estado': None, 'detalle': None}
            try:
                crs = leer_crs(ruta, tipo)
                if crs is None:
                    reg['estado'] = 'Sin CRS'
                else:
                    epsg = identificar_epsg(crs)
                    reg['crs_nombre'] = crs.name
                    reg['epsg'] = epsg
                    reg['estado'] = 'OK' if es_objetivo(crs, epsg) else 'Reproyectar'
            except Exception as e:
                reg['estado'] = 'Error'
                reg['detalle'] = str(e)[:200]
            registros.append(reg)

    df = pd.DataFrame(registros, columns=['carpeta', 'archivo', 'tipo', 'ruta', 'crs_nombre',
                                          'epsg', 'estado', 'detalle'])
    df['epsg'] = df['epsg'].astype('Int64')
    return df

## 4. Inventario

In [ ]:
%%time
inventario = inventariar(RAIZ)

print(f'Capas encontradas: {len(inventario):,}  '
      f'({(inventario["tipo"] == "vector").sum()} vector, {(inventario["tipo"] == "raster").sum()} raster)\n')
print(inventario.groupby(['tipo', 'estado']).size().unstack(fill_value=0).to_string())

In [ ]:
# Resumen por sistema de referencia encontrado
(inventario.fillna({'crs_nombre': '(sin CRS)'})
           .groupby(['epsg', 'crs_nombre'], dropna=False)
           .size().rename('capas').reset_index()
           .sort_values('capas', ascending=False))

In [ ]:
# Detalle completo (filtra con inventario[inventario['estado'] == '...'] si es necesario)
pd.set_option('display.max_colwidth', 80)
inventario.drop(columns='ruta').sort_values(['estado', 'carpeta', 'archivo'])

In [ ]:
# Capas que requieren atención manual
revisar = inventario[inventario['estado'].isin(['Sin CRS', 'Error'])]
if revisar.empty:
    print('Todas las capas tienen CRS legible.')
else:
    print(f'{len(revisar)} capas sin CRS o con error de lectura (no se reproyectarán):')
    display(revisar[['carpeta', 'archivo', 'tipo', 'estado', 'detalle']])

## 5. Funciones de respaldo y reproyección

**Respaldo:** copia todos los archivos de la capa (p. ej. `.shp .shx .dbf .prj .cpg ...` o `.tif .tfw .aux.xml ...`) a `RUTA_RAIZ/_respaldo_reproyeccion/<misma estructura de carpetas>`. Si ya existe un respaldo de esa capa no se sobrescribe, para conservar siempre el original más antiguo.

**Vector:** se lee con GeoPandas, se reproyecta y se escribe en una carpeta temporal dentro de la misma carpeta; después se reemplazan los archivos originales (la escritura es atómica por archivo, de modo que un error a medio camino no deja la capa corrupta).

**Raster:** se calcula la nueva malla con `calculate_default_transform` y se reproyecta banda por banda conservando tipo de dato, NoData, compresión, paleta de colores y descripciones. Los formatos distintos de GeoTIFF (`.img`, `.asc`, `.jp2`) se escriben primero como GeoTIFF temporal y se copian al formato original.

In [ ]:
SIDECARS_VECTOR = ['.sbn', '.sbx', '.qix', '.fbn', '.fbx', '.ain', '.aih', '.atx', '.ixs', '.mxs']
SIDECARS_RASTER = ['.tfw', '.tifw', '.tiffw', '.wld', '.j2w', '.rrd']


def archivos_de_capa(ruta):
    '''Archivo principal + todos los que comparten su nombre base (componentes y auxiliares).'''
    ruta = Path(ruta)
    base = ruta.stem.lower()
    return [p for p in ruta.parent.iterdir()
            if p.is_file() and (p.stem.lower() == base or p.name.lower().startswith(ruta.name.lower() + '.'))]


def respaldar(ruta):
    ruta = Path(ruta)
    destino = RAIZ / CARPETA_RESPALDO / ruta.parent.relative_to(RAIZ)
    destino.mkdir(parents=True, exist_ok=True)
    for p in archivos_de_capa(ruta):
        if not (destino / p.name).exists():
            shutil.copy2(p, destino / p.name)


def eliminar_sidecars(ruta, extensiones_stem, extensiones_nombre=()):
    ruta = Path(ruta)
    for ext in extensiones_stem:
        for p in (ruta.with_suffix(ext), ruta.with_suffix(ext.upper())):
            if p.exists():
                p.unlink()
    for ext in extensiones_nombre:
        p = ruta.with_name(ruta.name + ext)
        if p.exists():
            p.unlink()


def reproyectar_vector(ruta):
    ruta = Path(ruta)
    kwargs = {'encoding': ENCODING_SHP} if ENCODING_SHP else {}
    gdf = gpd.read_file(ruta, **kwargs)
    n_original = len(gdf)
    gdf = gdf.to_crs(CRS_OBJETIVO)

    tmp_dir = ruta.parent / f'.tmp_reproy_{ruta.stem}'
    tmp_dir.mkdir(exist_ok=True)
    try:
        gdf.to_file(tmp_dir / ruta.name, driver='ESRI Shapefile', encoding='UTF-8')
        n_nuevo = pyogrio.read_info(tmp_dir / ruta.name)['features']
        if n_nuevo != n_original:
            raise RuntimeError(f'Se escribieron {n_nuevo} de {n_original} registros')
        for p in tmp_dir.iterdir():
            os.replace(p, ruta.parent / p.name)   # sobrescribe .shp .shx .dbf .prj .cpg
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    eliminar_sidecars(ruta, SIDECARS_VECTOR)
    return f'{n_original:,} registros'


def reproyectar_raster(ruta):
    ruta = Path(ruta)
    tmp = ruta.with_name(f'.tmp_reproy_{ruta.stem}.tif')

    with rasterio.open(ruta) as src:
        driver = src.driver
        transform, ancho, alto = calculate_default_transform(
            src.crs, CRS_OBJETIVO, src.width, src.height, *src.bounds)

        entero = np.issubdtype(np.dtype(src.dtypes[0]), np.integer)
        try:
            colormap = src.colormap(1)
        except ValueError:
            colormap = None
        resampling = RESAMPLING_ENTEROS if (entero or colormap) else RESAMPLING_FLOTANTES

        nodata = src.nodata
        if nodata is None and not entero:
            # flotantes sin NoData: las celdas fuera de la imagen quedan como NaN (GeoTIFF)
            # o -9999 (otros formatos, p. ej. .asc, donde NaN no es bien soportado)
            nodata = np.nan if driver == 'GTiff' else -9999.0

        perfil = src.profile.copy()
        perfil.update(crs=CRS_OBJETIVO, transform=transform, width=ancho, height=alto,
                      nodata=nodata, BIGTIFF='IF_SAFER')
        if driver != 'GTiff':
            for k in ('blockxsize', 'blockysize', 'tiled', 'interleave'):
                perfil.pop(k, None)
            perfil.update(driver='GTiff', compress='lzw')

        with rasterio.open(tmp, 'w', **perfil) as dst:
            for b in range(1, src.count + 1):
                reproject(source=rasterio.band(src, b), destination=rasterio.band(dst, b),
                          src_transform=src.transform, src_crs=src.crs, src_nodata=src.nodata,
                          dst_transform=transform, dst_crs=CRS_OBJETIVO, dst_nodata=nodata,
                          resampling=resampling)
            if colormap:
                dst.write_colormap(1, colormap)
            dst.descriptions = src.descriptions
            dst.update_tags(**src.tags())

    # Reemplazar el original (el archivo fuente ya está cerrado)
    eliminar_sidecars(ruta, SIDECARS_RASTER, ['.aux.xml', '.ovr', '.msk'])
    if driver == 'GTiff':
        os.replace(tmp, ruta)
    else:
        try:
            ruta.unlink()
            rasterio.shutil.copy(tmp, ruta, driver=driver)
        finally:
            tmp.unlink(missing_ok=True)
            Path(str(tmp) + '.aux.xml').unlink(missing_ok=True)

    return f'{ancho}x{alto} px, {resampling.name}'

## 6. Reproyección

Solo se ejecuta si `SOLO_INVENTARIO = False`. Procesa únicamente las capas con estado `Reproyectar`.

In [ ]:
%%time
pendientes = inventario[inventario['estado'] == 'Reproyectar']
resultados = []

if SOLO_INVENTARIO:
    print(f'Modo SOLO_INVENTARIO: no se modificó nada. {len(pendientes)} capas por reproyectar.')
    print('Cambia SOLO_INVENTARIO = False en los parámetros para ejecutar la reproyección.')
elif pendientes.empty:
    print('No hay capas por reproyectar.')
else:
    for i, r in enumerate(pendientes.itertuples(), 1):
        etiqueta = f'[{i}/{len(pendientes)}] {r.carpeta}/{r.archivo}  (EPSG {r.epsg} -> {EPSG_OBJETIVO})'
        try:
            if HACER_RESPALDO:
                respaldar(r.ruta)
            fn = reproyectar_vector if r.tipo == 'vector' else reproyectar_raster
            detalle = fn(r.ruta)
            resultados.append({'ruta': r.ruta, 'resultado': 'Reproyectada', 'detalle_proceso': detalle})
            print(f'{etiqueta}  OK — {detalle}')
        except Exception as e:
            resultados.append({'ruta': r.ruta, 'resultado': 'Falló', 'detalle_proceso': str(e)[:200]})
            print(f'{etiqueta}  ERROR — {e}')

    res = pd.DataFrame(resultados)
    print(f'\nReproyectadas: {(res["resultado"] == "Reproyectada").sum()}   '
          f'Fallidas: {(res["resultado"] == "Falló").sum()}')

## 7. Verificación

In [ ]:
if SOLO_INVENTARIO:
    print('Modo SOLO_INVENTARIO: no hay nada que verificar.')
else:
    verificacion = inventariar(RAIZ)
    print(verificacion.groupby(['tipo', 'estado']).size().unstack(fill_value=0).to_string())
    pend = verificacion[verificacion['estado'] == 'Reproyectar']
    if not pend.empty:
        print('\nCapas que siguen fuera del EPSG objetivo:')
        display(pend[['carpeta', 'archivo', 'epsg', 'crs_nombre']])

## 8. Exportar reporte

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
reporte = inventario.copy()
if resultados:
    reporte = reporte.merge(pd.DataFrame(resultados), on='ruta', how='left')

marca = datetime.now().strftime('%Y%m%d_%H%M%S')
ruta_reporte = f'{OUTPUT_DIR}/reporte_crs_{marca}.csv'
reporte.to_csv(ruta_reporte, index=False, encoding='utf-8-sig')   # utf-8-sig: abre bien en Excel
print(f'Reporte: {ruta_reporte}')